# Fase 5: Purificação de Features e Neutralização Fatorial

Este notebook demonstra:
1. **Clustered Feature Importance (CFI)**: Mitigação do efeito de substituição entre variáveis altamente correlacionadas e classificação em `Informative`, `Redundant` e `Noise`.
2. **Neutralização Fatorial em Duas Etapas**:
   - *Etapa 1 (Setorial)*: Regressão linear OLS contra dummies de setor $\rightarrow$ Extração do Resíduo 1.
   - *Etapa 2 (Size Bias)*: Regressão quadrática contra $\ln(\text{Market Cap})$ e $[\ln(\text{Market Cap})]^2 \rightarrow$ Extração do Resíduo Purificado.
3. **Diagnóstico de Multicolinearidade (VIF - Variance Inflation Factor)**.

In [1]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath('..'))

from src.features.purification import (
    compute_vif,
    select_informative_features,
    neutralize_factors,
)

In [2]:
# 1. Teste de VIF (Variance Inflation Factor)
np.random.seed(42)
x1 = np.random.normal(0, 1, 200)
x2 = np.random.normal(0, 1, 200)
x3 = 1.9 * x1 + np.random.normal(0, 0.05, 200) # Quase linear com x1

df_sample = pd.DataFrame({"Factor_A": x1, "Factor_B": x2, "Factor_A_Clone": x3})
vif_results = compute_vif(df_sample)
vif_results

In [3]:
# 2. Neutralização Fatorial em Duas Etapas (Setor + Size)
tickers = [f"STOCK_{i}" for i in range(100)]
sectors = pd.Series(np.random.choice(["Tech", "Finance", "Energy", "Health"], size=100), index=tickers)
mcap = pd.Series(np.random.uniform(1e9, 1e11, size=100), index=tickers)

# Feature bruta com viés de setor e market cap
raw_feat = 2.5 * (sectors == "Tech").astype(float) + 0.5 * np.log(mcap) + np.random.normal(0, 1, 100)
df_raw = pd.DataFrame({"raw_factor": raw_feat}, index=tickers)

df_purified = neutralize_factors(df_raw, sectors, mcap)
print("Resíduos Fatoriais Purificados (Primeiras 5 linhas):")
print(df_purified.head())